# Station availability in the Gyeongju region, 2010–2026

**Goal.** Before any long detection run for the new high-quality Gyeongju catalog, establish *which stations
exist in the region, when they operated, and for which of them we already hold waveform data locally.*

**Two different kinds of "available" — kept separate throughout:**
1. **Metadata availability** — the station *operated* (from inventories/epoch tables).
2. **Local data availability** — waveforms are *already on this disk*. **This means exactly: found in one of
   the archive directories listed in §4** (`NS/`, `GJ/`, the NECIS download archive, and the stage-1
   `KS_KG/continuous/` archive). Data stored anywhere else are invisible to this notebook and will show as
   "no local data" — if that happens for data you know you have, add the directory to the scan list in §4.

| network | what it is | metadata source | local waveforms |
|---|---|---|---|
| **KS** | KMA permanent | **`KS_KG_metadata_1.0.2.xml`** (real channel epochs, incl. retired codes) | NECIS + stage-1 `KS_KG/continuous/` |
| **KG** | KIGAM permanent | **StationXML** (63 stations, channel epochs — authoritative) | NECIS downloads |
| **NS** | GHBSN dense network | `GHBSN_info_ver202312_modified.csv` (244 stations, epochs + coords) | `NS/` SDS archive |
| **GJ / PK / SS / TP** | 2016 Gyeongju M5.8 aftershock temporary arrays | `GJ/station_table/gj_temporary_station_list.csv` (coords; no epoch table → operating period read from the data itself) | `GJ/` SDS archive (Sep 2016 – Feb 2017) |

**What "epoch" means here (plain language).** An *epoch* is simply the period during which a station (or one
of its instrument set-ups) was actually operating: an install date and a removal/close date. Networks publish
these in their metadata because stations get installed, moved, upgraded, and retired. We need them because a
station that *exists in a coordinate list* was not necessarily *recording* in, say, 2012 — and if we later count
"how many stations could have detected this event in 2012", using today's station list would silently overcount.
When a sensor is swapped or reoriented at the same site, the metadata splits into multiple epochs, and the
instrument response can differ between them — which matters once we compute magnitudes. For the temporary GJ
arrays there is no published epoch table, so we take the honest substitute: the first-to-last day of actual data
on disk.

**Known honesty limits (also printed where relevant):**
- KS and KG epochs are now authoritative (channel-level, from the StationXML), including retired codes; the old
  `station_update.dat` + used-lists (no epochs) are no longer used.
- The GJ/PK/SS/TP coordinates must be recovered from the deployment report/paper.

Edit only the **parameters** cell, then run top-to-bottom.

In [ ]:
# ---------------------------------------------------------------- parameters (edit here)
import os, re, glob
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.dates as mdates, matplotlib.font_manager as fm
import pygmt, xml.etree.ElementTree as ET
from collections import defaultdict

GYEONGJU = (35.856, 129.224)      # reference point (Gyeongju city); 2016 M5.8 epicentre ~ (35.77, 129.19)
RMAX_KM  = 100.0                  # keep stations within this distance of Gyeongju
SPAN     = ("2010-01-01", "2026-12-31")

ROOT  = "/home/msseo/works/02.Ulsan_Fault_detection"
NECIS = "/home/msseo/works/Claude/data/necis/continuous"

def resolve(*cands):
    "First existing path among candidates (handles pre-/post-cleanup layouts transparently)."
    for c in cands:
        if os.path.exists(c): return c
    raise FileNotFoundError(cands)
P_XML = resolve(f"{ROOT}/KS_KG/local_magnitudes/responses/master/KS_KG_metadata_1.0.2.xml",
                f"{ROOT}/stage1_KS_KG_exploration/KS_KG/local_magnitudes/responses/master/KS_KG_metadata_1.0.2.xml")
P_NSCSV = resolve(f"{ROOT}/GHBSN_metadata/20240715/GHBSN_info_ver202312_modified.csv")
P_KSDAT = resolve(f"{ROOT}/KS_KG/station_table/station_update.dat",
                  f"{ROOT}/stage1_KS_KG_exploration/KS_KG/station_table/station_update.dat")
P_KSYRS = os.path.dirname(P_KSDAT)                       # stations_2015.csv .. stations_2024.csv live beside it
P_GJCSV = resolve(f"{ROOT}/GJ/station_table/gj_temporary_station_list.csv")
P_ST1   = resolve(f"{ROOT}/KS_KG/continuous",
                  f"{ROOT}/stage1_KS_KG_exploration/KS_KG/continuous")   # stage-1 KS+KG continuous SDS archive
P_FAULT = resolve(f"{ROOT}/KS_KG/HypoInv/faults_lonlat.gmt",
                  f"{ROOT}/stage1_KS_KG_exploration/KS_KG/HypoInv/faults_lonlat.gmt")

_av={f.name for f in fm.fontManager.ttflist}
for _f in ("Helvetica","Arial","Nimbus Sans","TeX Gyre Heros","DejaVu Sans"):
    if _f in _av: mpl.rcParams["font.family"]=_f; break
mpl.rcParams.update({"figure.dpi":130,"axes.grid":True,"grid.alpha":0.3,"font.size":10,
                     "legend.framealpha":1,"legend.edgecolor":"black","legend.facecolor":"white",
                     "axes.unicode_minus":False})
def hav_km(lat1,lon1,lat2,lon2):
    a=(np.sin(np.radians(lat2-lat1)/2)**2 +
       np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(np.radians(lon2-lon1)/2)**2)
    return 2*6371.0*np.arcsin(np.sqrt(a))
T0,T1=pd.Timestamp(SPAN[0]),pd.Timestamp(SPAN[1]); TODAY=pd.Timestamp.today().normalize()
ACCEL=("G","N"); NSXML="{http://www.fdsn.org/xml/station/1}"
def _ts(x):
    try: return pd.Timestamp(x).tz_localize(None)
    except Exception: return None
def load_stationxml(path,accel=ACCEL):
    A=defaultdict(lambda:{"lat":None,"lon":None,"seis":[],"all":[],"bands":set()}); net=None
    for ev,el in ET.iterparse(path,events=("start","end")):
        t=el.tag.split("}")[-1]
        if ev=="start" and t=="Network": net=el.get("code")
        elif ev=="end" and t=="Station":
            d=A[(net,el.get("code"))]; la=el.findtext(NSXML+"Latitude")
            if la: d["lat"]=float(la); d["lon"]=float(el.findtext(NSXML+"Longitude"))
            for c in el.findall(NSXML+"Channel"):
                cc=c.get("code") or ""; d["bands"].add(cc[:2]); iv=(c.get("startDate"),c.get("endDate"))
                d["all"].append(iv)
                if len(cc)>1 and cc[1] not in accel: d["seis"].append(iv)
            el.clear()
    rows=[]
    for (net,code),d in A.items():
        if d["lat"] is None: continue
        use=d["seis"] or d["all"]
        st=[x for x in (_ts(a) for a,b in use if a) if x is not None]
        en=[x for x in (_ts(b) for a,b in use if b) if x is not None]
        me=max(en) if en else None
        rows.append(dict(net=net,sta=code,lat=d["lat"],lon=d["lon"],
                         t_start=min(st) if st else T0,
                         t_end=(TODAY if (me is None or me.year>=2098) else me),
                         accel_only=(len(d["seis"])==0),chan="/".join(sorted(d["bands"])),src="StationXML"))
    return pd.DataFrame(rows)
_XML=load_stationxml(P_XML); _XML["dist_km"]=hav_km(GYEONGJU[0],GYEONGJU[1],_XML.lat,_XML.lon); _XML=_XML[_XML.dist_km<=RMAX_KM]
print("paths resolved:"); [print("  ",p) for p in (P_XML,P_NSCSV,P_GJCSV,P_ST1,P_FAULT,NECIS)]

## 1 · KG (KIGAM) — StationXML epochs (authoritative)

Real **channel-level** operating epochs from `KS_KG_metadata_1.0.2.xml`; window from seismometer channels
(accelerometers `?G`/`?N`, left open to 2099, excluded); open end clipped to today.

In [ ]:
kg=_XML[_XML.net=="KG"].copy()
kg["dist_km"]=hav_km(GYEONGJU[0],GYEONGJU[1],kg.lat,kg.lon)
print(f"KG: {len(kg)} stations in inventory; {int((kg.dist_km<=RMAX_KM).sum())} within {RMAX_KM:.0f} km of Gyeongju")
kg[kg.dist_km<=RMAX_KM].sort_values("dist_km").head(10)

## 2 · NS (GHBSN dense network) — epoch table

`endtime = 2099` means the epoch is open; clipped to the span end and flagged.

In [ ]:
ns=pd.read_csv(P_NSCSV)
ns=ns.rename(columns={"station":"sta","stla":"lat","stlo":"lon","stel":"elev","channel":"chan",
                      "network":"net"})
ns["t_start"]=pd.to_datetime(ns.starttime); ns["t_end"]=pd.to_datetime(ns.endtime)
ns["open_epoch"]=ns.t_end.dt.year>=2099
ns.loc[ns.open_epoch,"t_end"]=T1
ns=ns[["net","sta","lat","lon","elev","t_start","t_end","chan","open_epoch"]].drop_duplicates("sta")
ns["src"]="GHBSN_info_csv"
ns["dist_km"]=hav_km(GYEONGJU[0],GYEONGJU[1],ns.lat,ns.lon)
print(f"NS/GHBSN: {len(ns)} stations; {int((ns.dist_km<=RMAX_KM).sum())} within {RMAX_KM:.0f} km "
      f"({int(ns.open_epoch.sum())} open epochs clipped to {T1.date()})")
ns.sort_values("dist_km").head(6)

## 3 · KS (KMA permanent) — StationXML epochs (now authoritative)

KS now comes from the **same StationXML** as KG (`KS_KG_metadata_1.0.2.xml`) — real channel epochs, incl. retired
codes. This replaces the old `station_update.dat` + used-station lists (no epochs, current codes only), so retired
stations like **USN** (→2016-05-31, replaced by **USN2**) and **DAG** (→2010-11-01, → **DAG2**) are now
represented with their true operating windows. Accelerometer-only stations are kept and flagged (`accel_only`).

In [ ]:
ks=_XML[_XML.net=="KS"].copy()
print(f"KS: {len(ks)} within {RMAX_KM:.0f} km | velocity/seismometer {int((~ks.accel_only).sum())} | "
      f"accelerometer-only {int(ks.accel_only.sum())} (kept, flagged)")
for a,b in [("USN","USN2"),("DAG","DAG2")]:
    if (ks.sta==a).any() and (ks.sta==b).any():
        print(f"  {a}: {ks.loc[ks.sta==a,'t_start'].iloc[0].date()}..{ks.loc[ks.sta==a,'t_end'].iloc[0].date()}  ->  "
              f"{b}: {ks.loc[ks.sta==b,'t_start'].iloc[0].date()}..{ks.loc[ks.sta==b,'t_end'].iloc[0].date()}")
ks.sort_values("dist_km").head(8)[["sta","lat","lon","dist_km","t_start","t_end","accel_only"]]

## 4 · Local waveform archives — what is already on disk

Filename-only scans (no waveform reads) over **every archive we hold**:

| archive | format | what it contains |
|---|---|---|
| `NS/` | SDS | GHBSN dense network (HH channels; the V*/L*/G*/EX dirs are state-of-health, skipped) |
| `GJ/` | SDS | 2016 aftershock temporary arrays |
| `KS_KG/continuous/` | SDS | **stage-1 KS + KG continuous downloads** (HH + HG channels) |
| NECIS archive | `YYYY/STA/...` | ongoing KS daily downloads (`~/works/Claude/data/necis/`) |

Result per (network, station): the set of (year, day-of-year) with ≥ 1 file, merged across archives into one
`LOCAL` dictionary used by every later cell.

In [ ]:
def scan_sds(root,prefixes=("HH",)):
    out={}
    if not os.path.isdir(root): return out
    for sta in sorted(os.listdir(root)):
        sd=os.path.join(root,sta)
        if not os.path.isdir(sd): continue
        days=set()
        for ch in os.listdir(sd):
            if not ch.endswith(".D") or not ch.startswith(tuple(prefixes)): continue
            for f in os.scandir(os.path.join(sd,ch)):
                p=f.name.split(".")
                if len(p)>=7 and p[-2].isdigit() and p[-1].isdigit(): days.add((int(p[-2]),int(p[-1])))
        if days: out[sta]=days
    return out
def scan_necis(root):
    out={}
    if not os.path.isdir(root): return out
    for ydir in sorted(glob.glob(os.path.join(root,"20??"))):
        y=int(os.path.basename(ydir))
        for sd in os.scandir(ydir):
            if not sd.is_dir(): continue
            days=out.setdefault(sd.name,set())
            for f in os.scandir(sd.path):
                p=f.name.split(".")
                if len(p)>=5 and p[3].isdigit() and p[4].isdigit(): days.add((int(p[3]),int(p[4])))
    return out
DATA_NS=scan_sds(os.path.join(ROOT,"NS"))                    # HH only (other dirs are state-of-health)
DATA_GJ=scan_sds(os.path.join(ROOT,"GJ"),prefixes=("",))     # all channels for the temporary arrays
DATA_ST1=scan_sds(P_ST1,prefixes=("HH","HG","EL","SH"))      # stage-1 KS+KG continuous (velocity + strong-motion)
DATA_NECIS=scan_necis(NECIS)
# ---- unify into LOCAL[(net,sta)] = set of (year, doy); KS vs KG resolved by the KG inventory ----
KGSET=set(kg.sta); LOCAL={}
def _add(net,sta,days): LOCAL.setdefault((net,sta),set()).update(days)
for sta,d in DATA_NS.items(): _add("NS",sta,d)
for sta,d in DATA_GJ.items(): _add("GJ2016",sta,d)
for src in (DATA_NECIS,DATA_ST1):
    for sta,d in src.items(): _add("KG" if sta in KGSET else "KS",sta,d)
def _span(d): return f"{min(d)[0]}.{min(d)[1]:03d}..{max(d)[0]}.{max(d)[1]:03d}"
print(f"NS archive        : {len(DATA_NS)} stations | example span N001 {_span(DATA_NS['N001']) if 'N001' in DATA_NS else '-'}")
print(f"GJ archive        : {len(DATA_GJ)} temporary stations | years {sorted({y for v in DATA_GJ.values() for y,_ in v})}")
print(f"stage-1 continuous: {len(DATA_ST1)} stations | years {sorted({y for v in DATA_ST1.values() for y,_ in v})}")
print(f"NECIS archive     : {len(DATA_NECIS)} stations | years {sorted({y for v in DATA_NECIS.values() for y,_ in v})}")
print(f"-> LOCAL union: {len(LOCAL)} (network,station) entries; "
      f"KG with local data: {sum(1 for (n,_) in LOCAL if n=='KG')}")
# ---- data with NO metadata: never drop silently ----
_meta=set(ns.sta)
NOMETA=sorted(sta for sta in DATA_NS if sta not in _meta)
if NOMETA:
    print(f"\nnote: {len(NOMETA)} U*/Y* station dirs in the NS archive are the ULJIN / JEONBUK deployments —")
    print(f"      OUT of the Gyeongju region (user-confirmed); excluded from the master table.")

## 5 · Master table

One row per station within the region: coordinates, source, metadata epoch, local-data day count and span.
The GJ/PK/SS/TP arrays are kept **without coordinates** (flagged) so nothing is silently dropped.
Saved to `station_availability_master.csv`.

In [ ]:
def days_info(net,sta):
    d=LOCAL.get((net,sta))
    if not d: return 0,None,None
    def ts(t): return pd.Timestamp(f"{t[0]}-01-01")+pd.Timedelta(days=t[1]-1)
    return len(d),ts(min(d)),ts(max(d))
rows=[]
for _,r in kg[kg.dist_km<=RMAX_KM].iterrows():
    n,d0,d1=days_info("KG",r.sta)
    rows.append(dict(net="KG",sta=r.sta,lat=r.lat,lon=r.lon,dist_km=r.dist_km,src=r.src,
                     t_start=r.t_start,t_end=r.t_end,ndays_local=n,local_first=d0,local_last=d1))
for _,r in ns[ns.dist_km<=RMAX_KM].iterrows():
    n,d0,d1=days_info("NS",r.sta)
    rows.append(dict(net="NS",sta=r.sta,lat=r.lat,lon=r.lon,dist_km=r.dist_km,src=r.src,
                     t_start=r.t_start,t_end=r.t_end,ndays_local=n,local_first=d0,local_last=d1))
for _,r in ks.iterrows():
    n,d0,d1=days_info("KS",r.sta)
    rows.append(dict(net="KS",sta=r.sta,lat=r.lat,lon=r.lon,dist_km=r.dist_km,src=r.src,
                     t_start=r.t_start,t_end=r.t_end,ndays_local=n,local_first=d0,local_last=d1))
gjm=pd.read_csv(P_GJCSV,encoding="utf-8-sig").set_index("Code")     # user-provided coordinates (2024 list)
for sta,d in DATA_GJ.items():
    n,d0,d1=days_info("GJ2016",sta)
    if sta in gjm.index:
        la,lo=float(gjm.loc[sta,"Latitude"]),float(gjm.loc[sta,"Longitude"])
        rows.append(dict(net="GJ2016",sta=sta,lat=la,lon=lo,dist_km=hav_km(GYEONGJU[0],GYEONGJU[1],la,lo),
                         src="gj_temporary_station_list.csv + SDS scan",
                         t_start=d0,t_end=d1,ndays_local=n,local_first=d0,local_last=d1))
    else:
        print(f"WARNING: {sta} has data but no coordinates in {os.path.basename(P_GJCSV)}")
        rows.append(dict(net="GJ2016",sta=sta,lat=np.nan,lon=np.nan,dist_km=np.nan,src="SDS scan (coords missing)",
                         t_start=d0,t_end=d1,ndays_local=n,local_first=d0,local_last=d1))
M=pd.DataFrame(rows).sort_values(["net","dist_km","sta"]).reset_index(drop=True)
out=os.path.join(ROOT,"Gyeongju_catalog","station_availability_master.csv")
M.to_csv(out,index=False)
print(M.groupby("net").agg(n=("sta","count"),with_local_data=("ndays_local",lambda x:(x>0).sum())).to_string())
print(f"\nwrote {out}  ({len(M)} stations)")

## 6 · Map — stations within the region, coloured by network

Includes the 2016 GJ/PK/SS/TP temporary arrays (coordinates from `gj_temporary_station_list.csv`).

In [ ]:
pad=0.35
reg=[GYEONGJU[1]-pad-0.35,GYEONGJU[1]+pad+0.35,GYEONGJU[0]-pad-0.3,GYEONGJU[0]+pad+0.3]
fig=pygmt.Figure()
with pygmt.config(MAP_FRAME_TYPE="plain",FORMAT_GEO_MAP="ddd.xx"):
    fig.basemap(region=reg,projection="M15c",
                frame=[f"WSne+tStations within {RMAX_KM:.0f} km of Gyeongju (2010-2026 metadata)","xa0.2f0.1","ya0.2f0.1"])
    fig.coast(shorelines="0.6p,black",resolution="f",water="230/242/250",land="255/253/248")
    if os.path.exists(P_FAULT): fig.plot(data=P_FAULT,pen="0.7p,gray40")
    th=np.linspace(0,2*np.pi,181)                                  # RMAX ring
    fig.plot(x=GYEONGJU[1]+RMAX_KM/111.32/np.cos(np.radians(GYEONGJU[0]))*np.cos(th),
             y=GYEONGJU[0]+RMAX_KM/110.57*np.sin(th),pen="0.8p,gray50,--")
    mm=M[M.lat.notna()]
    for net,colr,sty in [("NS","#2ca02c","c0.13c"),("KS","#1f77b4","t0.26c"),("KG","#ff7f0e","s0.22c"),
                         ("GJ2016","#d62728","i0.24c")]:
        s=mm[mm.net==net]
        if len(s): fig.plot(x=s.lon,y=s.lat,style=sty,fill=colr,pen="0.3p,black",label=f"{net} (n={len(s)})")
    fig.plot(x=[GYEONGJU[1]],y=[GYEONGJU[0]],style="a0.5c",fill="red",pen="0.8p,black",label="Gyeongju")
    fig.basemap(map_scale="jBL+w20k+o0.5c/0.5c+c35.85")
    fig.legend(position="JTR+jTR+o0.2c",box="+gwhite+p0.6p,black")
fig.show(width=900)
print(f"GJ2016 = the {int((M.net=='GJ2016').sum())} temporary aftershock-array stations (inverted red triangles; "
      f"operated Sep 2016 - Feb 2017 only).")

## 7 · Availability timeline (Gantt), 2010–2026

Light bar = **metadata epoch** (all networks now from real epochs — KS/KG StationXML, NS GHBSN);
dark bar = **local waveform data** on disk. NS stations are drawn as thin lines (many stations).

In [ ]:
fig,ax=plt.subplots(figsize=(12.5,13))
y=0; ylbl=[]
def spans_from_days(days):
    "compress a set of (yr,doy) into contiguous date spans"
    if not days: return []
    ds=sorted(pd.Timestamp(f"{a}-01-01")+pd.Timedelta(days=b-1) for a,b in days)
    out=[[ds[0],ds[0]]]
    for d in ds[1:]:
        if (d-out[-1][1]).days<=1: out[-1][1]=d
        else: out.append([d,d])
    return out
def bar(t0,t1,yy,color,h=0.72,alpha=1.0,z=2):
    if pd.isna(t0) or pd.isna(t1): return
    ax.barh(yy,(t1-t0).days,left=t0,height=h,color=color,alpha=alpha,zorder=z,lw=0)
for _,r in M[M.net=="GJ2016"].sort_values("sta").iterrows():
    for s0,s1 in spans_from_days(LOCAL.get(("GJ2016",r.sta),set())): bar(s0,s1+pd.Timedelta(days=1),y,"#d62728")
    ylbl.append(f"GJ {r.sta}"); y+=1
for _,r in M[M.net=="KG"].sort_values("dist_km",ascending=False).iterrows():
    bar(max(r.t_start,T0),min(r.t_end,T1),y,"#ffd8a8")
    for s0,s1 in spans_from_days(LOCAL.get(("KG",r.sta),set())): bar(s0,s1+pd.Timedelta(days=1),y,"#ff7f0e",h=0.45,z=3)
    ylbl.append(f"KG {r.sta}"); y+=1
for _,r in M[M.net=="KS"].sort_values("dist_km",ascending=False).iterrows():
    bar(r.t_start,r.t_end,y,"#c4d8f0")
    for s0,s1 in spans_from_days(LOCAL.get(("KS",r.sta),set())): bar(s0,s1+pd.Timedelta(days=1),y,"#1f77b4",h=0.45,z=3)
    ylbl.append(f"KS {r.sta}"); y+=1
y0ns=y
for _,r in M[M.net=="NS"].sort_values("t_start").iterrows():
    bar(max(r.t_start,T0),min(r.t_end,T1),y,"#c8e6c9",h=0.9)
    for s0,s1 in spans_from_days(LOCAL.get(("NS",r.sta),set())): bar(s0,s1+pd.Timedelta(days=1),y,"#2ca02c",h=0.55,z=3)
    y+=1
ax.text(T0,(y0ns+y)/2,f"  NS x {int((M.net=='NS').sum())} (thin rows, sorted by start)",va="center",fontsize=9,color="#1b5e20")
ax.set_yticks(range(len(ylbl)),ylbl,fontsize=6.5)
ax.set_ylim(-1,y); ax.set_xlim(T0,T1)
ax.xaxis.set_major_locator(mdates.YearLocator(2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set(title=f"Station availability within {RMAX_KM:.0f} km of Gyeongju — light: metadata epoch, dark: local data on disk")
plt.tight_layout(); plt.show()

## 8 · Operating-station count vs time

Solid = stations whose **metadata epoch** covers the month; dashed = stations with **local data** that month.
All curves use real metadata epochs (KS/KG from StationXML, NS from GHBSN).

In [ ]:
months=pd.date_range(T0,T1,freq="MS")
def month_key(ts): return (ts.year,ts.month)
loc_months={}
for (net,sta),days in LOCAL.items():
    s=loc_months.setdefault((net,sta),set())
    for a,b in days: s.add(month_key(pd.Timestamp(f"{a}-01-01")+pd.Timedelta(days=b-1)))
fig,ax=plt.subplots(figsize=(11,4.6))
for net,colr in [("KS","#1f77b4"),("KG","#ff7f0e"),("NS","#2ca02c"),("GJ2016","#d62728")]:
    sub=M[M.net==net]
    meta=[int(((sub.t_start<=m)&(sub.t_end>=m)).sum()) for m in months]
    ax.plot(months,meta,color=colr,lw=1.6,label=f"{net} metadata")
    stas=set(sub.sta)
    loc=[sum(1 for (n2,s2),mm in loc_months.items() if n2==net and s2 in stas and month_key(m) in mm) for m in months]
    ax.plot(months,loc,color=colr,lw=1.2,ls="--")
ax.set(xlabel="year",ylabel="operating stations",title=f"Stations within {RMAX_KM:.0f} km of Gyeongju (solid: metadata; dashed: local data)")
ax.legend(fontsize=8.5,ncol=4); plt.tight_layout(); plt.show()

## 9 · Summary and next steps

In [ ]:
print("="*110)
print(f"STATION AVAILABILITY, {RMAX_KM:.0f} km AROUND GYEONGJU, 2010-2026".center(110)); print("="*110)
S=M.groupby("net").agg(n_stations=("sta","count"),
                       with_local_data=("ndays_local",lambda x:int((x>0).sum())),
                       local_days_total=("ndays_local","sum"),
                       first_meta=("t_start","min"),last_meta=("t_end","max"))
print(S.to_string())
print("-"*110)
print("TAKE-HOMES")
print(f" * KG: authoritative StationXML epochs; NS/GHBSN: {int((M.net=='NS').sum())} dense stations (epochs from the info CSV).")
print(f" * KS: coordinates known for all, but epochs are only LOWER-BOUNDED by the 2015-2024 used lists.")
print(f" * Local data: NS archive + NECIS downloads + the 2016 GJ aftershock arrays ({int((M.net=='GJ2016').sum())} stations, Sep 2016-Feb 2017,")
print(f"   coordinates from gj_temporary_station_list.csv; operating period = data span, no published epochs).")
print("NEXT")
print(" 1. Query NECIS station metadata for true KS epochs 2010-2026 (fills the pre-2015 gap).")
print(" 2. Decide the detection station set per era (e.g. pre-2016 sparse KS/KG vs 2016 aftershock arrays vs post-2017 dense +NS).")
print(" 3. Fill the local-data gaps -> see 02.Local_data_gap_analysis.ipynb, then plan the NECIS bulk download.")